@author: Timothy Chee Cheng Lui, https://github.com/timmehlui/

GeoChemNet is an interact tool for visualizing and interpreting outliers in geochemical data using networks. More details of the creation of the networks and protocols for use can be found in the accompanying paper, "GeoChemNet: An Interactive Tool for Visualizing and Interpreting Outliers in Geochemical Data Using Networks"
https://doi.org/10.1016/j.apgeochem.2026.106712

GeoChemNet visualizes complex multidimensional data through a network/graph made of nodes and edges. The bipartite network is built from two kinds of nodes: elemental nodes for each element, and sample nodes for each analyte. Sample nodes are connected to elemental nodes using a spring, and the spring strength depends on concentration, i.e. a sample with very high Al will be pulled really close to the Al node. GeoChemNet aims to interpret the results of the equilibrium position of all these nodes after the balance of all the springs's pushing and pulling.

In [ ]:
# Imports. Run this cell at the start.
from pyvis.network import Network
import networkx as nx
from networkx.algorithms import bipartite
import pandas as pd
import pyrolite.comp
import scipy
import matplotlib.pyplot as plt
from matplotlib import colors
from sklearn.preprocessing import MinMaxScaler
import matplotlib as mpl
import matplotlib.cm as cm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from ipywidgets import interact, widgets
import os
import imageio

# QT is needed to create interactive visualizations in a pop-up window
%matplotlib qt 

In [ ]:
# Functions. Run this cell at the start.

def make_network(data_og, element_list, meta_list, id_column_name, threshold, 
                                 weight_exponent, k_multiplier, network_style, remove_univariate, seed):
    """
    Creates a network graph from geochemical data.
    
    Parameters:
    -----------
    data_og : pd.DataFrame
        Original data containing elements and metadata
    element_list : list
        List of compositional element columns to be analyzed
    meta_list : list
        List of metadata columns to preserve
    id_column_name : str
        Name of the column containing sample IDs
    threshold : float
        Cutoff value for determining outliers
    weight_exponent : float
        Exponent to apply to edge weights
    k_multiplier : float
        Multiplier for spring layout spacing
    network_style : str
        Either 'scaling' or 'percentile' for thresholding method
    remove_univariate : bool
        Whether to remove univariate relations
    seed : int
        Random seed for consistent layouts
        
    Returns:
    --------
    G : nx.Graph
        NetworkX graph object
    pos : dict
        Node positions for graph layout
    data_processed : pd.DataFrame
        Processed data with CLR transformation and scaling
    """
    # Subsets the data into two parts
    data_elems = data_og[element_list].copy()
    data_meta = data_og[meta_list].copy()
    
    # Apply CLR transformation only to elemental side
    clr_data = data_elems.pyrocomp.CLR()
    for name in clr_data.columns:
        new_name = name.split('(')[1].split('/')[0]
        dict_name = {name : new_name}
        clr_data = clr_data.rename(columns = dict_name)
    
    # Scale CLR from 0 - 1, removing issues with negative numbers and bringing all elements to equal relative weighting
    scaler = MinMaxScaler()
    data_clr = pd.DataFrame(scaler.fit_transform(clr_data), columns=clr_data.columns)
    
    # Recombine Data
    data_recomb = pd.concat([data_meta, data_clr], axis=1)
    data_processed = data_recomb.copy()

    # Create networks based on thresholding style
    G = nx.Graph()
    if network_style == 'scaling':
        for i in data_processed.itertuples():
            for j in range(len(meta_list), data_processed.shape[1]):
                if i[j+1] >= threshold or i[j+1] <= 1 - threshold:
                    G.add_edge(i[1], data_processed.columns[j], weight = i[j+1] ** weight_exponent)
    elif network_style == 'percentile':
        for j in range(len(meta_list), data_processed.shape[1]):
            upper_limit = data_processed[data_processed.columns[j]].quantile(1-threshold)
            lower_limit = data_processed[data_processed.columns[j]].quantile(threshold)
            for i in data_processed.itertuples():
                if i[j+1] >= upper_limit or i[j+1] <= lower_limit:
                    G.add_edge(i[1], data_processed.columns[j], weight = i[j+1] ** weight_exponent)
    else:
        print('Network Style not found, please adjust')
        return None, None, None

    # Optionally ignore univariate relations.
    if remove_univariate:
        nodes_to_remove = [node for node in G.nodes if G.degree(node) == 1]
        G.remove_nodes_from(nodes_to_remove)

    # Calculates the position of nodes after reaching spring equilibrium
    pos = nx.spring_layout(G, seed = seed, k = k_multiplier * G.number_of_nodes() ** -0.5)

    return(G, pos, data_processed)

def plot_figure_with_map(G, pos, data_processed, element_list, id_column_name, 
                            x_coord, y_coord, weight_exponent, xlim=None, ylim=None, label_sample_nodes=None):
    """
    Creates an interactive figure with network and geospatial map.
    
    Parameters:
    -----------
    G : nx.Graph
        NetworkX graph object
    pos : dict
        Node positions for graph layout
    data_processed : pd.DataFrame
        Processed data with CLR transformation and scaling
    element_list : list
        List of elements available for coloring
    id_column_name : str
        Name of the column containing sample IDs
    x_coord : str
        Name of column for X coordinates
    y_coord : str
        Name of column for Y coordinates
    weight_exponent : float
        Exponent used in edge weights (for reversing transformation)
    xlim : tuple, optional
        X-axis limits for network plot as (xmin, xmax). Default is None (auto).
    ylim : tuple, optional
        Y-axis limits for network plot as (ymin, ymax). Default is None (auto).
    label_sample_nodes : bool
        Should all sample nodes be labeled with text or not
        
    Returns:
    --------
    interact widget
        Interactive widget with dropdowns and text input
    """
    
    # Add text interactable for complex selecting
    fig = plt.figure(figsize=(64,32))

    def plot_variable(variable_color, select_for_text):
        elem_color = variable_color
        select_for_text = select_for_text
        
        # Process text
        if '.' in select_for_text:
            text_combos = select_for_text.split(',')
        
            plt.clf()
            
            data_processed.loc[:, 'selected'] = 2
            
            ax1 = fig.add_axes([0.025, 0.1, 0.5, 0.8])  # Define the axes width of figures

            # For coloring of figures, choose what the minimum and maximum color are going to be
            vmin = 0.2
            vmax = 0.8
            cmap = cm.coolwarm
                
            # Advanced code: In my case, we have 7 clusters so altering the color scale to fit that coloring.
            if variable_color == 'cluster label':
                vmin = 0
                vmax = 6
                cmap = cm.turbo
            
            # Create network colors and attributes
            for i in G.nodes():
                if type(i) == int:
                    G.nodes[i]['attr'] = data_processed.loc[data_processed[id_column_name] == i, elem_color]

            color_map = []
            outline_map = []
            line_width_map = []
            labeldict = {}
            node_size_list = []
            norm = mpl.colors.Normalize(vmin = vmin, vmax=vmax)
            m = cm.ScalarMappable(norm = norm, cmap = cmap)

            for node in G:
                if type(node) == str:
                    color_map.append('red')
                    labeldict[node] = node
                    node_size_list.append(2000)
                    outline_map.append('black')
                    line_width_map.append(0)
                else:
                    color_map.append(mpl.colors.rgb2hex(m.to_rgba(G.nodes[node]['attr'])))
                    if label_sample_nodes == True:
                        labeldict[node] = node
                    node_size_list.append(100)
                    outline_map.append('black')

                    # Selection logic
                    for combo in text_combos:
                        highlow = combo.split()[0]
                        elem_select = combo.split()[1].split('.')[0]

                        if int(data_processed.loc[data_processed[id_column_name] == node, 'selected']) >= 1:
                            if highlow == 'high':
                                if G.has_edge(node, elem_select) and G.get_edge_data(elem_select, node)['weight'] ** (1/weight_exponent) > 0.5:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 1
                                else:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 0

                            if highlow == 'low':
                                if G.has_edge(node, elem_select) and G.get_edge_data(elem_select, node)['weight'] ** (1/weight_exponent) < 0.5:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 1
                                else:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 0

                    if int(data_processed.loc[data_processed[id_column_name] == node, 'selected']) == 1:
                        line_width_map.append(2)
                    else:
                        line_width_map.append(0)

            # Draw network
            if label_sample_nodes == True: # If labeling sample nodes, want all text smaller
                nx.draw_networkx(G, pos = pos, node_color = color_map, edgecolors = outline_map, 
                               linewidths = line_width_map, labels = labeldict, 
                               node_size = node_size_list, with_labels = True, 
                               width = 0.1, font_size = 10)
            else: # If not labeling sample nodes, then just make the elemental node text bigger.
                nx.draw_networkx(G, pos = pos, node_color = color_map, edgecolors = outline_map, 
                               linewidths = line_width_map, labels = labeldict, 
                               node_size = node_size_list, with_labels = True, 
                               width = 0.1, font_size = 30)
                
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin = vmin, vmax=vmax))
            sm._A = []
            cbar = plt.colorbar(sm)
            tick_font_size = 15
            cbar.ax.tick_params(labelsize=tick_font_size)
            ax1.set_title('Network', fontsize=20)

            # Apply zoom if specified
            if xlim is not None:
                ax1.set_xlim(xlim)
            if ylim is not None:
                ax1.set_ylim(ylim)
            
            # Figure 2 - Geospatial Map
            ax2 = fig.add_axes([0.575, 0.1, 0.3, 0.8])

            # Add selected attribute to dataset
            data_processed['selected'] = data_processed['selected'].fillna(0)

            # Draw map
            ax2.scatter(data_processed[x_coord], data_processed[y_coord], c = data_processed[elem_color], 
                       cmap = cmap, norm=plt.Normalize(vmin = vmin, vmax=vmax), 
                       marker = 'o', s=20)
            ax2.scatter(data_processed[data_processed['selected'] == 1][x_coord], 
                       data_processed[data_processed['selected'] == 1][y_coord], 
                       facecolors = 'none', edgecolors='black', 
                       linewidth = 1, marker = 'o',  s=20)

            ax2.tick_params(axis='y', labelsize=15)
            ax2.tick_params(axis='x', labelsize=15)
            ax2.set_xlabel('Eastings (m)', fontsize=15)
            ax2.set_ylabel('Northings (m)', fontsize=15)
            ax2.legend(["Colored by \n" + elem_color , "Selected\nin Network"], fontsize = 15)
            ax2.set_title('Geospatial Map', fontsize=20)

            fig.suptitle(' Network and Geospatial Map colored by ' + elem_color + 
                        ' selecting for ' + select_for_text, fontsize=22)
            
            plt.draw()
    
    # Create widget options
    element_list_copy = element_list.copy()
    element_list_copy.append('cluster label') # Advanced code: Want to color network by metadata
        
    dropdown_color = widgets.Dropdown(options=element_list_copy,
                                description='Choose variable to color by:',
                                layout={'width': 'max-content'},
                                style={'description_width': 'initial'})

    selection_text = widgets.Textarea(value='high Al',
                                      description='What do you want to select for? Formatted "(high/low) Element":',
                                      layout={'width': 'max-content'},
                                      style={'description_width': 'initial'})

    return interact(plot_variable, variable_color=dropdown_color, select_for_text = selection_text)

#### Editing to test the, add sample_node_name feature

def plot_figure_no_map(G, pos, data_processed, element_list, id_column_name, weight_exponent, xlim=None, ylim=None, label_sample_nodes=None):
    """
    Creates an interactive figure with network visualization only (no geospatial map).
    
    Parameters:
    -----------
    G : nx.Graph
        NetworkX graph object
    pos : dict
        Node positions for graph layout
    data_processed : pd.DataFrame
        Processed data with CLR transformation and scaling
    element_list : list
        List of elements available for coloring
    id_column_name : str
        Name of the column containing sample IDs
    weight_exponent : float
        Exponent used in edge weights (for reversing transformation)
    xlim : tuple, optional
        X-axis limits for network plot as (xmin, xmax). Default is None (auto).
    ylim : tuple, optional
        Y-axis limits for network plot as (ymin, ymax). Default is None (auto).
    label_sample_nodes : bool
        Should all sample nodes be labeled with text or not
        
    Returns:
    --------
    interact widget
        Interactive widget with dropdowns and text input
    """
    
    # Create interactive figure
    fig = plt.figure(figsize=(64,32))

    def plot_variable(variable_color, select_for_text):
        elem_color = variable_color
        select_for_text = select_for_text
        
        # Process text
        if '.' in select_for_text:
            text_combos = select_for_text.split(',')
        
            plt.clf()
            
            data_processed.loc[:, 'selected'] = 2
            
            ax1 = fig.add_axes([0.025, 0.1, 0.95, 0.8])  # Define the axes width of figures

            vmin = 0.1
            vmax = 0.9
            cmap = cm.coolwarm

            # Assign attributes to nodes
            for i in G.nodes():
                if i not in element_list:  # Find only the sample nodes, which are integers
                    G.nodes[i]['attr'] = data_processed.loc[data_processed[id_column_name] == i, elem_color]

            color_map = []
            outline_map = []
            line_width_map = []
            labeldict = {}
            node_size_list = []
            norm = mpl.colors.Normalize(vmin = vmin, vmax=vmax)
            m = cm.ScalarMappable(norm = norm, cmap = cmap)

            for node in G:
                if node in element_list:
                    color_map.append('red')
                    labeldict[node] = node
                    node_size_list.append(2000)
                    outline_map.append('black')
                    line_width_map.append(0)
                else:
                    color_map.append(mpl.colors.rgb2hex(m.to_rgba(G.nodes[node]['attr'])))
                    if label_sample_nodes == True:
                        labeldict[node] = node
                    node_size_list.append(100)
                    outline_map.append('black')

                    # Selection logic
                    # Strategy: assign all selected 1, and then slowly start turning things into 0
                    for combo in text_combos:
                        highlow = combo.split()[0]
                        elem_select = combo.split()[1].split('.')[0]

                        if int(data_processed.loc[data_processed[id_column_name] == node, 'selected']) >= 1:
                            if highlow == 'high':
                                if G.has_edge(node, elem_select) and G.get_edge_data(elem_select, node)['weight'] ** (1/weight_exponent) > 0.5:
                                    # Keep track in dataset
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 1
                                else:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 0

                            if highlow == 'low':
                                if G.has_edge(node, elem_select) and G.get_edge_data(elem_select, node)['weight'] ** (1/weight_exponent) < 0.5:
                                    # Keep track in dataset
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 1
                                else:
                                    data_processed.loc[data_processed[id_column_name] == node, 'selected'] = 0

                    if int(data_processed.loc[data_processed[id_column_name] == node, 'selected']) == 1:
                        line_width_map.append(2)
                    else:
                        line_width_map.append(0)

            # Apply zoom if specified
            if xlim is not None:
                ax1.set_xlim(xlim)
            if ylim is not None:
                ax1.set_ylim(ylim)
                        
            # Draw network
            if label_sample_nodes == True: # If labeling sample nodes, want all text smaller
                nx.draw_networkx(G, pos = pos, node_color = color_map, edgecolors = outline_map, 
                               linewidths = line_width_map, labels = labeldict, 
                               node_size = node_size_list, with_labels = True, 
                               width = 0.1, font_size = 10)
            else: # If not labeling sample nodes, then just make the elemental node text bigger.
                nx.draw_networkx(G, pos = pos, node_color = color_map, edgecolors = outline_map, 
                               linewidths = line_width_map, labels = labeldict, 
                               node_size = node_size_list, with_labels = True, 
                               width = 0.1, font_size = 30)
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin = vmin, vmax=vmax))
            sm._A = []
            cbar = plt.colorbar(sm)
            tick_font_size = 15
            cbar.ax.tick_params(labelsize=tick_font_size)
            ax1.set_title('Network', fontsize=20)
            
            fig.suptitle(' Network colored by ' + elem_color + ' selecting for ' + select_for_text, fontsize=30)

            plt.draw()
    
    # Create widget options
    dropdown_color = widgets.Dropdown(options=element_list,
                                description='Choose variable to color by:',
                                layout={'width': 'max-content'},
                                style={'description_width': 'initial'})

    selection_text = widgets.Textarea(value='high ' + element_list[0],
                                      description='What do you want to select for? Formatted "(high/low) Element":',
                                      layout={'width': 'max-content'},
                                      style={'description_width': 'initial'})

    return interact(plot_variable, variable_color=dropdown_color, select_for_text = selection_text)

In [ ]:
############################# START EDITING HERE #############################
#### Import Dataset
## Dataset file loctation and name in your computer
data_og = pd.read_csv("/Users/timmehlui/Documents/geochemicalData.csv")

In [ ]:
#### Network Parameters

## Please edit the following parameters accordingly
# List of compositional elements you want to analyze in your dataset
element_list = ['U', 'V', 'Mo', 'Cr', 'Co', 'Ni', 'Cu', 'Pb', 'Zn', 'Al', 'La', 'Nb', 'Sc', 'Sn', 'Ti', 'Zr']
element_list.sort()

# List of meta data that you still want to have access for, eg, Sample IDs, coordinate systems
meta_list = ['ID', 'east_rotate_zero', 'north_rotate_zero', 'cluster label']

# Name of the column that represents sample IDs
id_column_name = 'ID'

# Name of columns that represent X-Y coordinates (If using mapping tool)
x_coord = 'east_rotate_zero'
y_coord = 'north_rotate_zero'

# Threshold represents the cut off to determine what is or isn't an outlier. We recommend starting small and then consider decreasing.
# Optimal threshold value also depends on number of samples 
threshold = 0.85

# Parameters to tweak network distances.
weight_exponent = 1
k_multiplier = 1

# We offer two ways to create networks, each with pros and cons. For more details read the paper. Please un-comment your choice
network_style = 'scaling' # For scaling based thresholding
#network_style = 'percentile' # For percentile based thresholding

# If you want to focus on multivariate relations and remove the noise of univariate anomalies, make sure this is true
remove_univariate = False

# Random seed, in order for you to create consistent network representations
seed = 8

# Specify axes for x-lim and y-lim in network space for zoom in in the figure plot
xlim=(-0.9, 0.9)
ylim=(-0.9, 0.9)

# Do you want to label sample nodes?
label_sample_nodes = False

In [ ]:
# Main execution
G, pos, data_processed = make_network(data_og, element_list, meta_list, id_column_name, 
                            threshold, weight_exponent, k_multiplier, 
                            network_style, remove_univariate, seed)

# Option 1: With map
plot_figure_with_map(G, pos, data_processed, element_list, id_column_name, x_coord, y_coord, weight_exponent, xlim, ylim, label_sample_nodes)

# Option 2: Without map
#plot_figure_no_map(G, pos, data_processed, element_list, id_column_name, weight_exponent, xlim, ylim, label_sample_nodes)